# DEC model-result figures

This notebook is the editable, figure-by-figure version of `02_model_result.py`.
It reads existing model outputs only and does not retrain the model.

Run the cells from top to bottom. Each plotting cell begins with its own font,
figure-size, spacing, margin, resolution, and output-format parameters. The
supplementary DEC-loss panel (d) and reconstruction-audit panel (e) are drawn
and saved as separate figures.


In [ ]:
from __future__ import annotations

import csv
import re
import zipfile
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple, Union

import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

# The first path supports normal use from the notebook directory. The fallback
# keeps the notebook runnable when Jupyter was opened from another directory.
NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "02_model_result.py").exists():
    NOTEBOOK_DIR = Path(
        r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
RESULT_ROOT = PROJECT_ROOT / "uilt_oneday_all" / "02clustering" / "result"
AE_DIR = RESULT_ROOT / "AE"
DEC_DIR = RESULT_ROOT / "DEC_result"
K_DIR = RESULT_ROOT / "Figure" / "optimal_cluster"
BEST_K_FILE = RESULT_ROOT / "Figure" / "summary_best_k.xlsx"
OUTPUT_DIR = NOTEBOOK_DIR / "Fig2_cluster_results" / "submission"

EVENT = "event5"
GAMMA = 0.1
TIME_RANGE_S = (0.0, 60.0)
FREQUENCY_RANGE_HZ = (0.0, 50.0)

COLORS = {
    "blue": "#0F4D92",
    "blue_mid": "#5B7FCA",
    "blue_soft": "#B4C0E4",
    "orange": "#D97706",
    "lilac": "#9A7CC1",
    "neutral_light": "#D8D8D8",
    "neutral_mid": "#8F8F8F",
    "neutral_dark": "#4D4D4D",
    "black": "#272727",
}

CLUSTER_COLORS = [
    "#0072B2", "#E69F00", "#009E73", "#CC79A7",
    "#56B4E9", "#D55E00", "#F0E442", "#000000",
]
K_METRIC_COLUMNS = {"WCSS": "WCSS"}


def publication_style(
    *,
    font_family="Arial",
    font_size=10.0,
    axes_label_size=10.0,
    axes_title_size=10.5,
    tick_label_size=9.3,
    legend_size=9.2,
):
    """Return editable publication typography settings for one figure cell."""
    return {
        "font.family": "sans-serif",
        "font.sans-serif": [font_family, "Helvetica", "DejaVu Sans", "sans-serif"],
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "font.size": font_size,
        "axes.labelsize": axes_label_size,
        "axes.titlesize": axes_title_size,
        "xtick.labelsize": tick_label_size,
        "ytick.labelsize": tick_label_size,
        "legend.fontsize": legend_size,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.75,
        "xtick.major.width": 0.65,
        "ytick.major.width": 0.65,
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "legend.frameon": False,
        "savefig.facecolor": "white",
    }


def save_notebook_figure(fig, output_base, formats=("svg", "pdf", "tiff", "png"), tiff_dpi=600):
    """Save a figure without closing it, so the notebook can display it."""
    output_base = Path(output_base)
    output_base.parent.mkdir(parents=True, exist_ok=True)
    saved = []
    for requested in formats:
        extension = requested.lower().lstrip(".")
        if extension == "tif":
            extension = "tiff"
        path = output_base.with_suffix(f".{extension}")
        kwargs = {"bbox_inches": "tight", "facecolor": "white"}
        if extension == "png":
            kwargs["dpi"] = 300
        elif extension == "tiff":
            kwargs["dpi"] = tiff_dpi
            kwargs["pil_kwargs"] = {"compression": "tiff_lzw"}
        fig.savefig(path, **kwargs)
        saved.append(path)
    return saved


## Shared data-loading and calculation functions


In [ ]:
def event_number(value: Union[str, Path]) -> int:
    """Return the numeric part of an event name for natural sorting."""
    match = re.search(r"event(\d+)", str(value))
    if match is None:
        raise ValueError(f"Cannot extract event number from {value!r}")
    return int(match.group(1))


def read_csv_numeric(path: Path) -> List[Dict[str, Union[float, str]]]:
    """Read a CSV and convert numeric fields to floats."""
    with path.open("r", newline="", encoding="utf-8-sig") as handle:
        rows: List[Dict[str, Union[float, str]]] = []
        for row in csv.DictReader(handle):
            clean: Dict[str, Union[float, str]] = {}
            for key, value in row.items():
                try:
                    clean[key] = float(value)
                except (TypeError, ValueError):
                    clean[key] = value
            rows.append(clean)
    return rows


def _column_index(cell_ref: str) -> int:
    letters = re.match(r"([A-Z]+)", cell_ref)
    if letters is None:
        raise ValueError(f"Invalid XLSX cell reference: {cell_ref}")
    value = 0
    for char in letters.group(1):
        value = value * 26 + ord(char) - ord("A") + 1
    return value - 1


def _shared_text(node: ET.Element, ns: Mapping[str, str]) -> str:
    return "".join(item.text or "" for item in node.findall(".//m:t", ns))


def read_xlsx_sheet(path: Path, sheet_name: str = "Sheet1") -> List[Dict[str, object]]:
    """Read a small XLSX worksheet without requiring pandas/openpyxl."""
    ns = {
        "m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "rel": "http://schemas.openxmlformats.org/package/2006/relationships",
    }
    with zipfile.ZipFile(path) as archive:
        shared: List[str] = []
        if "xl/sharedStrings.xml" in archive.namelist():
            root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
            shared = [_shared_text(node, ns) for node in root.findall("m:si", ns)]

        workbook = ET.fromstring(archive.read("xl/workbook.xml"))
        relationships = ET.fromstring(archive.read("xl/_rels/workbook.xml.rels"))
        targets = {
            rel.attrib["Id"]: rel.attrib["Target"]
            for rel in relationships.findall("rel:Relationship", ns)
        }
        target = None
        for sheet in workbook.findall(".//m:sheet", ns):
            if sheet.attrib["name"] == sheet_name:
                target = targets[sheet.attrib[f"{{{ns['r']}}}id"]]
                break
        if target is None:
            raise ValueError(f"Worksheet {sheet_name!r} not found in {path}")

        target = target.lstrip("/")
        sheet_xml = target if target.startswith("xl/") else f"xl/{target}"
        worksheet = ET.fromstring(archive.read(sheet_xml))

    table: List[List[object]] = []
    for row in worksheet.findall(".//m:sheetData/m:row", ns):
        values: List[object] = []
        for cell in row.findall("m:c", ns):
            index = _column_index(cell.attrib["r"])
            while len(values) <= index:
                values.append("")
            cell_type = cell.attrib.get("t")
            if cell_type == "inlineStr":
                values[index] = _shared_text(cell, ns)
                continue
            value_node = cell.find("m:v", ns)
            if value_node is None:
                values[index] = ""
                continue
            raw = value_node.text or ""
            if cell_type == "s":
                values[index] = shared[int(raw)]
            else:
                try:
                    values[index] = float(raw)
                except ValueError:
                    values[index] = raw
        table.append(values)

    if not table:
        return []
    header = [str(value) for value in table[0]]
    records: List[Dict[str, object]] = []
    for values in table[1:]:
        padded = (values + [""] * len(header))[: len(header)]
        records.append(dict(zip(header, padded)))
    return records


def stack_curves(curves: Sequence[np.ndarray]) -> np.ndarray:
    """Pad unequal-length curves with NaN so event summaries remain aligned."""
    if not curves:
        raise ValueError("No curves were provided")
    max_length = max(len(curve) for curve in curves)
    array = np.full((len(curves), max_length), np.nan, dtype=float)
    for row, curve in enumerate(curves):
        array[row, : len(curve)] = curve
    return array


def moving_average(values: np.ndarray, window: int) -> np.ndarray:
    """Centered moving average with edge padding (no artificial end collapse)."""
    values = np.asarray(values, dtype=float)
    if window <= 1 or len(values) < window:
        return values.copy()
    if window % 2 == 0:
        window += 1
    half = window // 2
    padded = np.pad(values, (half, half), mode="edge")
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(padded, kernel, mode="valid")


def add_panel_label(ax: plt.Axes, label: str, x: float = -0.16, y: float = 1.10) -> None:
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        fontsize=9,
        fontweight="bold",
        ha="left",
        va="bottom",
    )


def cluster_color(cluster_id: int) -> str:
    return CLUSTER_COLORS[cluster_id % len(CLUSTER_COLORS)]


def load_ae_curves() -> Tuple[List[str], np.ndarray, np.ndarray]:
    paths = sorted(
        AE_DIR.glob("event*_stft_maxmin_Z_pretrain_log.csv"),
        key=event_number,
    )
    if not paths:
        raise FileNotFoundError(f"No AE logs found in {AE_DIR}")
    events: List[str] = []
    training: List[np.ndarray] = []
    validation: List[np.ndarray] = []
    for path in paths:
        rows = read_csv_numeric(path)
        events.append(re.search(r"event\d+", path.name).group(0))
        training.append(np.asarray([row["loss"] for row in rows], dtype=float))
        validation.append(np.asarray([row["val_loss"] for row in rows], dtype=float))
    return events, stack_curves(training), stack_curves(validation)


def load_k_data() -> Tuple[List[Tuple[str, np.ndarray, np.ndarray]], Dict[str, Dict[str, int]]]:
    paths = sorted(K_DIR.glob("event*.xlsx"), key=event_number)
    if not paths:
        raise FileNotFoundError(f"No WCSS files found in {K_DIR}")

    curves: List[Tuple[str, np.ndarray, np.ndarray]] = []
    for path in paths:
        rows = read_xlsx_sheet(path)
        k_values = np.asarray([row["n_clusters"] for row in rows], dtype=float)
        wcss = np.asarray([row["WCSS"] for row in rows], dtype=float)
        curves.append((path.stem, k_values, wcss))

    summary: Dict[str, Dict[str, int]] = {}
    for row in read_xlsx_sheet(BEST_K_FILE):
        event = str(row["Event"])
        summary[event] = {
            short_name: int(float(row[column]))
            for short_name, column in K_METRIC_COLUMNS.items()
        }
    return curves, summary


def compute_cluster_quality(npz_paths: Sequence[Path]) -> List[Dict[str, Union[float, int, str]]]:
    """Compute cluster quality in the final 32-dimensional DEC embedding."""
    metrics: List[Dict[str, Union[float, int, str]]] = []
    for path in sorted(npz_paths, key=event_number):
        with np.load(path, allow_pickle=True) as archive:
            features = np.asarray(archive["feature"], dtype=np.float64)
            labels = np.asarray(archive["y_pred_last"], dtype=int)
        unique = np.unique(labels)
        if len(unique) < 2 or len(unique) >= len(labels):
            silhouette = np.nan
            calinski = np.nan
            davies = np.nan
        else:
            silhouette = float(silhouette_score(features, labels, metric="euclidean"))
            calinski = float(calinski_harabasz_score(features, labels))
            davies = float(davies_bouldin_score(features, labels))
        metrics.append(
            {
                "event": re.search(r"event\d+", path.name).group(0),
                "n_samples": int(len(labels)),
                "n_clusters": int(len(unique)),
                "silhouette": silhouette,
                "calinski_harabasz": calinski,
                "davies_bouldin": davies,
            }
        )
    return metrics


def representative_indices(labels: np.ndarray, distances: np.ndarray) -> Dict[int, int]:
    """Select the observed sample nearest each DEC centroid (cluster medoid proxy)."""
    representatives: Dict[int, int] = {}
    for cluster_id in sorted(int(value) for value in np.unique(labels)):
        members = np.where(labels == cluster_id)[0]
        if distances.ndim == 2 and cluster_id < distances.shape[1]:
            chosen = members[np.argmin(distances[members, cluster_id])]
        else:
            chosen = members[len(members) // 2]
        representatives[cluster_id] = int(chosen)
    return representatives


def load_representative_event(event: str) -> Dict[str, object]:
    npz_path = DEC_DIR / "all_parement" / f"{event}_all_parement.npz"
    loss_path = DEC_DIR / "loss_csv" / f"{event}training_loss.csv"
    if not npz_path.exists():
        raise FileNotFoundError(npz_path)
    if not loss_path.exists():
        raise FileNotFoundError(loss_path)
    with np.load(npz_path, allow_pickle=True) as archive:
        data: Dict[str, object] = {
            "x_file": np.asarray(archive["x_file"]),
            "spectral": np.asarray(archive["spectral"]),
            "reconstructed": np.asarray(archive["respectral"]),
            "features": np.asarray(archive["feature"]),
            "tsne": np.asarray(archive["redu"]),
            "labels": np.asarray(archive["y_pred_last"], dtype=int),
            "n_clusters": int(np.asarray(archive["n_clusters"])),
            "distances": np.asarray(archive["distances"]),
        }
    data["loss_rows"] = read_csv_numeric(loss_path)
    data["representatives"] = representative_indices(data["labels"], data["distances"])
    return data


def load_dec_loss_curves() -> Dict[str, List[Dict[str, Union[float, str]]]]:
    """Load DEC optimization logs for all events in natural event order."""
    paths = sorted((DEC_DIR / "loss_csv").glob("event*training_loss.csv"), key=event_number)
    if not paths:
        raise FileNotFoundError(f"No DEC loss logs found in {DEC_DIR / 'loss_csv'}")
    return {
        re.search(r"event\d+", path.name).group(0): read_csv_numeric(path)
        for path in paths
    }


def summarize_curves(array: np.ndarray) -> Dict[str, np.ndarray]:
    return {
        "median": np.nanmedian(array, axis=0),
        "q25": np.nanpercentile(array, 25, axis=0),
        "q75": np.nanpercentile(array, 75, axis=0),
        "n": np.sum(np.isfinite(array), axis=0),
    }


def positive_for_log(values: np.ndarray) -> np.ndarray:
    finite_positive = values[np.isfinite(values) & (values > 0)]
    floor = np.nanmin(finite_positive) * 0.5 if len(finite_positive) else 1e-8
    return np.where(values > 0, values, floor)


## Shared plotting functions


In [ ]:
def draw_ae_summary(ax: plt.Axes, train: np.ndarray, validation: np.ndarray) -> None:
    epoch = np.arange(train.shape[1])
    train_summary = summarize_curves(train)
    val_summary = summarize_curves(validation)
    for summary, color, label in (
        (train_summary, COLORS["blue"], "Training"),
        (val_summary, COLORS["orange"], "Validation"),
    ):
        median = positive_for_log(summary["median"])
        lower = positive_for_log(summary["q25"])
        upper = positive_for_log(summary["q75"])
        ax.fill_between(epoch, lower, upper, color=color, alpha=0.18, linewidth=0)
        ax.plot(epoch, median, color=color, lw=1.35, label=label)
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Reconstruction MSE")
    ax.legend(loc="upper right", handlelength=1.3)


def draw_k_summary(
    ax_curve: plt.Axes,
    ax_hist: plt.Axes,
    curves: Sequence[Tuple[str, np.ndarray, np.ndarray]],
    best_k: Mapping[str, Mapping[str, int]],
    event: str,
) -> None:
    for current_event, k_values, wcss in curves:
        normalized = wcss / wcss[0]
        color = COLORS["blue"] if current_event == event else COLORS["neutral_mid"]
        alpha = 1.0 if current_event == event else 0.28
        width = 1.4 if current_event == event else 0.6
        ax_curve.plot(k_values, normalized, color=color, alpha=alpha, lw=width)
        chosen = best_k[current_event]["WCSS"]
        selected = np.where(k_values == chosen)[0]
        if len(selected):
            ax_curve.scatter(
                chosen,
                normalized[selected[0]],
                s=58 if current_event == event else 12,
                marker="*",
                color=COLORS["orange"],
                edgecolor=COLORS["black"] if current_event == event else "none",
                linewidth=0.35,
                alpha=1.0 if current_event == event else 0.65,
                zorder=4,
            )
    ax_curve.set_xlabel("Number of clusters, K")
    ax_curve.set_ylabel("Normalized WCSS")
#     ax_curve.set_title("Event-specific elbow selection")
    ax_curve.text(
        0.04,
        0.08,
        f"{event}: K={best_k[event]['WCSS']}",
        transform=ax_curve.transAxes,
        color=COLORS["blue"],
        fontweight="bold",
    )

    counts = Counter(values["WCSS"] for values in best_k.values())
    k_all = np.arange(min(counts), max(counts) + 1)
    heights = np.asarray([counts.get(int(k), 0) for k in k_all])
    bars = ax_hist.bar(k_all, heights, width=0.72, color=COLORS["blue_soft"], edgecolor=COLORS["blue"], linewidth=0.6)
    for bar, value in zip(bars, heights):
        if value:
            ax_hist.text(bar.get_x() + bar.get_width() / 2, value + 0.4, str(value), ha="center", va="bottom", fontsize=6)
    ax_hist.set_xlabel("K distribution")
    ax_hist.set_ylabel("Events")
#     ax_hist.set_title("K distribution")
    ax_hist.set_xticks(k_all)
    ax_hist.set_ylim(0, max(heights) * 1.18)


def draw_quality_summary(ax: plt.Axes, quality: Sequence[Mapping[str, object]], event: str) -> None:
    values = np.asarray([row["silhouette"] for row in quality], dtype=float)
    finite = np.isfinite(values)
    values = values[finite]
    rng = np.random.default_rng(20260812)
    x = rng.normal(1.0, 0.035, size=len(values))
    ax.boxplot(
        values,
        positions=[1],
        widths=0.34,
        patch_artist=True,
        showfliers=False,
        boxprops={"facecolor": COLORS["blue_soft"], "edgecolor": COLORS["blue"], "linewidth": 0.8},
        medianprops={"color": COLORS["black"], "linewidth": 1.0},
        whiskerprops={"color": COLORS["neutral_dark"], "linewidth": 0.8},
        capprops={"color": COLORS["neutral_dark"], "linewidth": 0.8},
    )
    ax.scatter(x, values, s=10, color=COLORS["neutral_dark"], alpha=0.55, edgecolor="none", rasterized=True)
    representative = next(row for row in quality if row["event"] == event)
    representative_value = float(representative["silhouette"])
    ax.scatter(1, representative_value, marker="*", s=72, color=COLORS["orange"], edgecolor=COLORS["black"], linewidth=0.4, zorder=5)
    ax.text(
        1.09,
        representative_value,
        f"{event}: {representative_value:.3f}",
        va="center",
        color=COLORS["orange"],
        fontweight="bold",
    )
    ax.set_xlim(0.72, 1.34)
    ax.set_xticks([1])
    ax.set_xticklabels(["44 events"])
    ax.set_ylabel("Silhouette coefficient")
#     ax.set_title("Cluster separation in final DEC space")



def draw_tsne(ax: plt.Axes, data: Mapping[str, object], event: str) -> None:
    embedding = np.asarray(data["tsne"])
    labels = np.asarray(data["labels"])
    representatives = data["representatives"]
    for cluster_id in sorted(int(value) for value in np.unique(labels)):
        mask = labels == cluster_id
        color = cluster_color(cluster_id)
        ax.scatter(
            embedding[mask, 0],
            embedding[mask, 1],
            s=5,
            alpha=0.48,
            color=color,
            edgecolor="none",
            rasterized=True,
        )
        center = np.median(embedding[mask], axis=0)
        label = ax.text(center[0], center[1], f"C{cluster_id}", ha="center", va="center", color="white", fontsize=6.5, fontweight="bold")
        label.set_path_effects([path_effects.withStroke(linewidth=1.6, foreground=color)])
    for cluster_id, index in representatives.items():
        ax.scatter(
            embedding[index, 0],
            embedding[index, 1],
            s=44,
            marker="*",
            color=cluster_color(cluster_id),
            edgecolor="white",
            linewidth=0.55,
            zorder=5,
        )
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")


def _squeezed(array: np.ndarray) -> np.ndarray:
    result = np.squeeze(np.asarray(array)).astype(float)
    if result.ndim != 2:
        raise ValueError(f"Expected a 2-D spectrogram, got shape {result.shape}")
    return result


def selected_spectrograms(data: Mapping[str, object]) -> Tuple[List[int], List[np.ndarray], List[np.ndarray], List[np.ndarray]]:
    representatives = data["representatives"]
    spectral = np.asarray(data["spectral"])
    reconstructed = np.asarray(data["reconstructed"])
    clusters = sorted(representatives)
    original = [_squeezed(spectral[representatives[cluster]]) for cluster in clusters]
    recon = [_squeezed(reconstructed[representatives[cluster]]) for cluster in clusters]
    residual = [np.abs(source - predicted) for source, predicted in zip(original, recon)]
    return clusters, original, recon, residual


def draw_spectrogram(
    ax: plt.Axes,
    spectrogram: np.ndarray,
    *,
    cmap: str,
    vmin: float,
    vmax: float,
    show_x: bool,
    show_y: bool,
) -> matplotlib.image.AxesImage:
    image = ax.imshow(
        spectrogram,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=[*TIME_RANGE_S, *FREQUENCY_RANGE_HZ],
        rasterized=True,
    )
    ax.set_xticks([0, 30, 60] if show_x else [])
    ax.set_yticks([0, 25, 50] if show_y else [])
    if show_x:
        ax.set_xlabel("Time (s)", labelpad=1.5)
    if show_y:
        ax.set_ylabel("Frequency (Hz)", labelpad=1.5)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.45)
        spine.set_color("0.35")
    return image


def cluster_frequency_profiles(
    data: Mapping[str, object],
) -> Tuple[np.ndarray, Dict[int, Dict[str, np.ndarray]]]:
    """Summarize time-averaged spectra within each DEC cluster."""
    spectral = np.asarray(data["spectral"])
    labels = np.asarray(data["labels"])
    frequencies = np.linspace(
        FREQUENCY_RANGE_HZ[0],
        FREQUENCY_RANGE_HZ[1],
        spectral.shape[1],
    )
    summaries: Dict[int, Dict[str, np.ndarray]] = {}
    for cluster_id in sorted(int(value) for value in np.unique(labels)):
        mask = labels == cluster_id
        # Each row is one one-minute window; average over time before the
        # across-window median/IQR so short, high-amplitude spikes do not
        # dominate the state-level profile.
        profiles = np.mean(
            spectral[mask, :, :, 0],
            axis=2,
            dtype=np.float32,
        )
        summaries[cluster_id] = {
            "median": np.median(profiles, axis=0),
            "q25": np.percentile(profiles, 25, axis=0),
            "q75": np.percentile(profiles, 75, axis=0),
        }
    return frequencies, summaries


def draw_frequency_profiles(ax: plt.Axes, data: Mapping[str, object]) -> None:
    frequencies, summaries = cluster_frequency_profiles(data)
    for cluster_id, summary in summaries.items():
        color = cluster_color(cluster_id)
        ax.fill_between(
            frequencies,
            summary["q25"],
            summary["q75"],
            color=color,
            alpha=0.14,
            linewidth=0,
        )
        ax.plot(
            frequencies,
            summary["median"],
            color=color,
            lw=1.25,
            label=f"C{cluster_id}",
        )
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Mean normalized amplitude")
    ax.set_xlim(FREQUENCY_RANGE_HZ)
    ax.legend(ncol=min(4, len(summaries)), loc="upper right", columnspacing=0.9, handlelength=1.3)


def draw_main_state_plate(
    spec_axes: Sequence[plt.Axes],
    data: Mapping[str, object],
) -> None:
    clusters, original, _, _ = selected_spectrograms(data)
    labels = np.asarray(data["labels"])
    pooled = np.concatenate([item.ravel() for item in original])
    vmin = max(0.0, float(np.nanpercentile(pooled, 1)))
    vmax = float(np.nanpercentile(pooled, 99))
    if vmax <= vmin:
        vmax = vmin + 1.0
    image = None
    for column, cluster_id in enumerate(clusters):
        ax = spec_axes[column]
        count = int(np.sum(labels == cluster_id))
        image = draw_spectrogram(
            ax,
            original[column],
            cmap="magma",
            vmin=vmin,
            vmax=vmax,
            show_x=True,
            show_y=column == 0,
        )
    add_panel_label(spec_axes[0], "e", x=-0.34, y=1.22)
  


def draw_individual_ae(ax: plt.Axes, array: np.ndarray, *, color: str, title: str) -> None:
    epoch = np.arange(array.shape[1])
    for curve in array:
        valid = np.isfinite(curve)
        ax.plot(epoch[valid], positive_for_log(curve[valid]), color=COLORS["neutral_mid"], alpha=0.22, lw=0.45)
    summary = summarize_curves(array)
    ax.plot(epoch, positive_for_log(summary["median"]), color=color, lw=1.4, label="Median")
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Reconstruction MSE")
#     ax.set_title(title)
    ax.legend(loc="upper right")


def draw_wcss_k_by_event(
    ax: plt.Axes,
    best_k: Mapping[str, Mapping[str, int]],
    representative_event: str,
) -> None:
    ordered_events = sorted(best_k, key=event_number)
    event_ids = np.asarray([event_number(event) for event in ordered_events])
    selected_k = np.asarray([best_k[event]["WCSS"] for event in ordered_events])

    ax.scatter(
        event_ids,
        selected_k,
        s=18,
        color=COLORS["blue_mid"],
        edgecolor="white",
        linewidth=0.35,
        alpha=0.85,
    )
    representative_id = event_number(representative_event)
    representative_k = best_k[representative_event]["WCSS"]
    ax.scatter(
        representative_id,
        representative_k,
        marker="*",
        s=82,
        color=COLORS["orange"],
        edgecolor=COLORS["black"],
        linewidth=0.4,
        zorder=5,
    )
    ax.text(
        representative_id + 1.2,
        representative_k,
        representative_event,
        color=COLORS["orange"],
        fontweight="bold",
        va="center",
    )
    unique_k = np.unique(selected_k)
    ax.set_xlabel("Event number")
    ax.set_ylabel("WCSS-selected K")
    ax.set_xticks([1, 10, 20, 30, 40, 44])
    ax.set_yticks(unique_k)
    ax.set_ylim(unique_k.min() - 0.45, unique_k.max() + 0.45)
#     ax.set_title("WCSS-selected K across events")
    ax.text(0.97, 0.07, f"n={len(ordered_events)} events", transform=ax.transAxes, ha="right", color=COLORS["neutral_dark"])


def prepare_dec_loss(loss_rows: Sequence[Mapping[str, object]]) -> Dict[str, np.ndarray]:
    total = np.asarray([row["total_loss"] for row in loss_rows], dtype=float)
    cluster_raw = np.asarray([row["cluster_loss"] for row in loss_rows], dtype=float)
    reconstruction = np.asarray([row["reconstruction_loss"] for row in loss_rows], dtype=float)
    cluster_weighted = GAMMA * cluster_raw
    return {
        "step": np.arange(len(total)),
        "total": total,
        "cluster_raw": cluster_raw,
        "cluster_weighted": cluster_weighted,
        "reconstruction": reconstruction,
    }


def summarize_dec_losses(
    loss_rows_by_event: Mapping[str, Sequence[Mapping[str, object]]],
    n_progress_points: int = 101,
) -> Dict[str, object]:
    """Align unequal DEC logs by relative progress and summarize across events.

    The first CSV record is an initialization diagnostic recorded before the
    first optimization update. It is omitted so that 0% denotes the first actual
    DEC update for every event.
    """
    progress = np.linspace(0.0, 100.0, n_progress_points)
    event_names = sorted(loss_rows_by_event, key=event_number)
    components = ("total", "cluster_weighted", "reconstruction")
    aligned = {
        component: np.full((len(event_names), n_progress_points), np.nan, dtype=float)
        for component in components
    }
    original_steps = np.zeros(len(event_names), dtype=int)

    for row_index, event in enumerate(event_names):
        prepared = prepare_dec_loss(loss_rows_by_event[event])
        original_steps[row_index] = len(prepared["step"])
        start_index = 1 if len(prepared["step"]) > 1 else 0
        source_progress = np.linspace(0.0, 100.0, len(prepared["step"]) - start_index)
        for component in components:
            values = np.asarray(prepared[component][start_index:], dtype=float)
            aligned[component][row_index] = np.interp(progress, source_progress, values)

    summaries = {component: summarize_curves(values) for component, values in aligned.items()}
    return {
        "progress": progress,
        "event_names": event_names,
        "original_steps": original_steps,
        "aligned": aligned,
        "summaries": summaries,
    }


def draw_dec_loss(ax: plt.Axes, loss_summary: Mapping[str, object]) -> None:
    progress = np.asarray(loss_summary["progress"], dtype=float)
    aligned = loss_summary["aligned"]
    summaries = loss_summary["summaries"]

    # Light individual total-loss trajectories expose event-level heterogeneity.
    for values in aligned["total"]:
        ax.plot(progress, values, color=COLORS["neutral_mid"], alpha=0.10, lw=0.38, zorder=1)

    series = [
        ("total", COLORS["blue"], "Total"),
        ("cluster_weighted", COLORS["orange"], rf"{GAMMA:g} $\times$ KL"),
        ("reconstruction", COLORS["lilac"], "Reconstruction"),
    ]
    for component, color, label in series:
        summary = summaries[component]
        ax.fill_between(
            progress,
            summary["q25"],
            summary["q75"],
            color=color,
            alpha=0.14,
            linewidth=0,
            zorder=2,
        )
        ax.plot(progress, summary["median"], color=color, lw=1.35, label=label, zorder=3)

    ax.set_xlabel("Normalized training progress (%)")
    ax.set_ylabel("Weighted loss")
#     ax.set_title("DEC convergence across events")
    ax.legend(loc="upper right", handlelength=1.2)
    ax.text(
        0.03,
        0.94,
        r"$L_{total}=0.1L_{KL}+L_{recon}$",
        transform=ax.transAxes,
        color=COLORS["neutral_dark"],
        ha="left",
        va="top",
    )

def pearson_correlation(first: np.ndarray, second: np.ndarray) -> float:
    first = np.asarray(first, dtype=float).ravel()
    second = np.asarray(second, dtype=float).ravel()
    first = first - np.mean(first)
    second = second - np.mean(second)
    denominator = np.sqrt(np.sum(first * first) * np.sum(second * second))
    if denominator <= 0:
        return np.nan
    return float(np.sum(first * second) / denominator)


def reconstruction_metrics(original: np.ndarray, reconstructed: np.ndarray) -> Dict[str, float]:
    residual = original - reconstructed
    original_variance = float(np.var(original))
    return {
        "rmse": float(np.sqrt(np.mean(residual * residual))),
        "pixel_correlation": pearson_correlation(original, reconstructed),
        "frequency_profile_correlation": pearson_correlation(
            np.mean(original, axis=1),
            np.mean(reconstructed, axis=1),
        ),
        "time_profile_correlation": pearson_correlation(
            np.mean(original, axis=0),
            np.mean(reconstructed, axis=0),
        ),
        "variance_retention": (
            float(np.var(reconstructed)) / original_variance
            if original_variance > 0
            else np.nan
        ),
    }


def draw_reconstruction_audit_plate(
    fig: plt.Figure,
    original_axes: Sequence[plt.Axes],
    reconstruction_axes: Sequence[plt.Axes],
    residual_axes: Sequence[plt.Axes],
    amplitude_colorbar_ax: plt.Axes,
    residual_colorbar_ax: plt.Axes,
    data: Mapping[str, object],
#     panel_label: str = "e",
) -> None:
    clusters, original, reconstructed, residual = selected_spectrograms(data)
    labels = np.asarray(data["labels"])
    amplitude_pool = np.concatenate(
        [values.ravel() for values in original + reconstructed]
    )
    amplitude_vmin = max(0.0, float(np.nanpercentile(amplitude_pool, 1)))
    amplitude_vmax = float(np.nanpercentile(amplitude_pool, 99))
    residual_vmax = float(
        np.nanpercentile(
            np.concatenate([values.ravel() for values in residual]),
            99,
        )
    )

    amplitude_image = None
    residual_image = None
    for column, cluster_id in enumerate(clusters):
        metrics = reconstruction_metrics(original[column], reconstructed[column])
        count = int(np.sum(labels == cluster_id))
        amplitude_image = draw_spectrogram(
            original_axes[column],
            original[column],
            cmap="magma",
            vmin=amplitude_vmin,
            vmax=amplitude_vmax,
            show_x=False,
            show_y=column == 0,
        )
        draw_spectrogram(
            reconstruction_axes[column],
            reconstructed[column],
            cmap="magma",
            vmin=amplitude_vmin,
            vmax=amplitude_vmax,
            show_x=False,
            show_y=column == 0,
        )
        residual_image = draw_spectrogram(
            residual_axes[column],
            residual[column],
            cmap="viridis",
            vmin=0.0,
            vmax=residual_vmax,
            show_x=True,
            show_y=column == 0,
        )
        residual_axes[column].set_xlabel("")

        reconstruction_axes[column].text(
            0.03,
            0.95,
            rf"$r_{{freq}}$={metrics['frequency_profile_correlation']:.2f}",
            transform=reconstruction_axes[column].transAxes,
            ha="left",
            va="top",
            fontsize=5.7,
            color="white",
            bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.45, "pad": 1.0},
        )

    residual_axes[-1].text(
        0.97,
        0.04,
        "Time (s)",
        transform=residual_axes[-1].transAxes,
        ha="right",
        va="bottom",
        fontsize=5.8,
        color="white",
        bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.35, "pad": 0.8},
    )

    row_specs = (
        (original_axes[0], "Original"),
        (reconstruction_axes[0], "Reconstruction"),
        (residual_axes[0], "|Residual|"),
    )
    for ax, label in row_specs:
        ax.text(
            -0.34,
            0.5,
            label,
            rotation=90,
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontweight="bold",
        )
#     add_panel_label(original_axes[0], panel_label, x=-0.34, y=1.25)

    amplitude_colorbar = fig.colorbar(
        amplitude_image,
        cax=amplitude_colorbar_ax,
        orientation="horizontal",
    )
    amplitude_colorbar.set_label("Normalized amplitude", labelpad=2)
    residual_colorbar = fig.colorbar(
        residual_image,
        cax=residual_colorbar_ax,
        orientation="horizontal",
    )
    residual_colorbar.set_label("|Residual|", labelpad=2)


## Load all model results once


In [ ]:
required_inputs = [AE_DIR, DEC_DIR, K_DIR, BEST_K_FILE]
missing_inputs = [path for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError("Missing required inputs: " + ", ".join(map(str, missing_inputs)))

events, train, validation = load_ae_curves()
k_curves, best_k = load_k_data()
npz_paths = sorted(
    (DEC_DIR / "all_parement").glob("event*_all_parement.npz"),
    key=event_number,
)
quality = compute_cluster_quality(npz_paths)
event_data = load_representative_event(EVENT)
dec_loss_rows = load_dec_loss_curves()
dec_loss_summary = summarize_dec_losses(dec_loss_rows)

print(
    f"Loaded {len(events)} AE logs, {len(k_curves)} WCSS curves, "
    f"{len(quality)} DEC result files, and {len(dec_loss_rows)} DEC loss logs."
)
print(
    f"Representative event: {EVENT}; K={event_data['n_clusters']}; "
    f"windows={len(event_data['labels'])}."
)


## Main-text Figure 2 (panels a–e)


In [ ]:
# ---------- Editable: colours, panel-label positions, typography, dimensions and spacing ----------
MAIN_COLORS = {
    "ae_train":        "#0F4D92",
    "ae_validation":   "#D97706",
    "k_curve_highlight": "#0F4D92",
    "k_curve_gray":      "#8F8F8F",
    "k_star":            "#D97706",
    "k_hist_bar":        "#B4C0E4",
    "k_hist_edge":       "#0F4D92",
    "quality_box":       "#B4C0E4",
    "quality_box_edge":  "#0F4D92",
    "quality_star":      "#D97706",
    "tsne_cluster_palette": [
        "#0072B2", "#E69F00", "#009E73", "#CC79A7",
        "#56B4E9", "#D55E00", "#F0E442", "#000000",
    ],
    "neutral_dark":      "#4D4D4D",
    "neutral_mid":       "#8F8F8F",
    "black":             "#272727",
    "spectrogram_cmap":  "magma",
}
MAIN_PANEL_LABEL_POS = {
    "a": dict(x=-0.14, y=1.02),
    "b": dict(x=-0.22, y=1.02),
    "c": dict(x=-0.14, y=1.02),
    "d": dict(x=-0.13, y=1.02),
    "e": dict(x=-0.27, y=1.12),
}
panel_font = 14
MAIN_FIGSIZE = (7.2, 6.65)
MAIN_FONT = dict(
    font_family="Arial",
    font_size=10.0,
    axes_label_size=10.0,
    axes_title_size=10.5,
    tick_label_size=9.3,
    legend_size=9.2,
)

MAIN_GRID = dict(
    height_ratios=[0.92, 1.0, 0.76],
    hspace=0.35,
    wspace=0.28,
)
MAIN_PANEL_B = dict(width_ratios=[1.45, 0.75], wspace=0.42)
MAIN_PANEL_E_WSPACE = 0.08
MAIN_MARGINS = dict(left=0.08, right=0.985, top=0.97, bottom=0.08)
OUTPUT_FORMATS = ("svg", "pdf", "tiff", "png")
TIFF_DPI = 600
TIME_RANGE_S = (0.0, 60.0)
FREQUENCY_RANGE_HZ = (0.0, 50.0)
# --------------------------------------------------------------------------------------------------------

def add_panel_label_local(ax, label, x=-0.16, y=1.10):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=panel_font, fontweight="bold", ha="left", va="bottom")

def cluster_color_local(cluster_id):
    return MAIN_COLORS["tsne_cluster_palette"][cluster_id % len(MAIN_COLORS["tsne_cluster_palette"])]

def summarize_curves_local(array):
    return {
        "median": np.nanmedian(array, axis=0),
        "q25": np.nanpercentile(array, 25, axis=0),
        "q75": np.nanpercentile(array, 75, axis=0),
        "n": np.sum(np.isfinite(array), axis=0),
    }

def positive_for_log_local(values):
    finite_positive = values[np.isfinite(values) & (values > 0)]
    floor = np.nanmin(finite_positive) * 0.5 if len(finite_positive) else 1e-8
    return np.where(values > 0, values, floor)

def draw_ae_summary_local(ax, train, validation):
    epoch = np.arange(train.shape[1])
    train_summary = summarize_curves_local(train)
    val_summary = summarize_curves_local(validation)
    for summary, color, label in (
        (train_summary, MAIN_COLORS["ae_train"], "Training"),
        (val_summary, MAIN_COLORS["ae_validation"], "Validation"),
    ):
        median = positive_for_log_local(summary["median"])
        lower = positive_for_log_local(summary["q25"])
        upper = positive_for_log_local(summary["q75"])
        ax.fill_between(epoch, lower, upper, color=color, alpha=0.18, linewidth=0)
        ax.plot(epoch, median, color=color, lw=1.35, label=label)
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Reconstruction MSE")
    ax.legend(loc="upper right", handlelength=1.3)

def draw_k_summary_local(ax_curve, ax_hist, curves, best_k, event):
    for current_event, k_values, wcss in curves:
        normalized = wcss / wcss[0]
        color = MAIN_COLORS["k_curve_highlight"] if current_event == event else MAIN_COLORS["k_curve_gray"]
        alpha = 1.0 if current_event == event else 0.28
        width = 1.4 if current_event == event else 0.6
        ax_curve.plot(k_values, normalized, color=color, alpha=alpha, lw=width)
        chosen = best_k[current_event]["WCSS"]
        selected = np.where(k_values == chosen)[0]
        if len(selected):
            ax_curve.scatter(
                chosen, normalized[selected[0]],
                s=58 if current_event == event else 12, marker="*",
                color=MAIN_COLORS["k_star"],
                edgecolor=MAIN_COLORS["black"] if current_event == event else "none",
                linewidth=0.35, alpha=1.0 if current_event == event else 0.65, zorder=4,
            )
    ax_curve.set_xlabel("Number of clusters, K")
    ax_curve.set_ylabel("Normalized WCSS")
    ax_curve.text(0.04, 0.08, f"{event}: K={best_k[event]['WCSS']}", transform=ax_curve.transAxes, color=MAIN_COLORS["k_curve_highlight"], fontweight="bold")
    from collections import Counter
    counts = Counter(values["WCSS"] for values in best_k.values())
    k_all = np.arange(min(counts), max(counts) + 1)
    heights = np.asarray([counts.get(int(k), 0) for k in k_all])
    bars = ax_hist.bar(k_all, heights, width=0.72, color=MAIN_COLORS["k_hist_bar"], edgecolor=MAIN_COLORS["k_hist_edge"], linewidth=0.6)
    for bar, value in zip(bars, heights):
        if value:
            ax_hist.text(bar.get_x() + bar.get_width() / 2, value + 0.4, str(value), ha="center", va="bottom", fontsize=6)
    ax_hist.set_xlabel("K distribution")
    ax_hist.set_ylabel("Events")
    ax_hist.set_xticks(k_all)
    ax_hist.set_ylim(0, max(heights) * 1.18)

def draw_quality_summary_local(ax, quality, event):
    values = np.asarray([row["silhouette"] for row in quality], dtype=float)
    finite = np.isfinite(values)
    values = values[finite]
    rng = np.random.default_rng(20260812)
    x = rng.normal(1.0, 0.035, size=len(values))
    ax.boxplot(
        values, positions=[1], widths=0.34, patch_artist=True, showfliers=False,
        boxprops={"facecolor": MAIN_COLORS["quality_box"], "edgecolor": MAIN_COLORS["quality_box_edge"], "linewidth": 0.8},
        medianprops={"color": MAIN_COLORS["black"], "linewidth": 1.0},
        whiskerprops={"color": MAIN_COLORS["neutral_dark"], "linewidth": 0.8},
        capprops={"color": MAIN_COLORS["neutral_dark"], "linewidth": 0.8},
    )
    ax.scatter(x, values, s=10, color=MAIN_COLORS["neutral_dark"], alpha=0.55, edgecolor="none", rasterized=True)
    representative = next(row for row in quality if row["event"] == event)
    representative_value = float(representative["silhouette"])
    ax.scatter(1, representative_value, marker="*", s=72, color=MAIN_COLORS["quality_star"], edgecolor=MAIN_COLORS["black"], linewidth=0.4, zorder=5)
    ax.text(1.09, representative_value, f"{event}: {representative_value:.3f}", va="center", color=MAIN_COLORS["quality_star"], fontweight="bold")
    ax.set_xlim(0.72, 1.34)
    ax.set_xticks([1])
    ax.set_xticklabels(["44 events"])
    ax.set_ylabel("Silhouette coefficient")

def draw_tsne_local(ax, data, event):
    embedding = np.asarray(data["tsne"])
    labels = np.asarray(data["labels"])
    representatives = data["representatives"]
    for cluster_id in sorted(int(value) for value in np.unique(labels)):
        mask = labels == cluster_id
        color = cluster_color_local(cluster_id)
        ax.scatter(embedding[mask, 0], embedding[mask, 1], s=5, alpha=0.48, color=color, edgecolor="none", rasterized=True)
        center = np.median(embedding[mask], axis=0)
        lbl = ax.text(center[0], center[1], f"C{cluster_id}", ha="center", va="center", color="white", fontsize=6.5, fontweight="bold")
        lbl.set_path_effects([path_effects.withStroke(linewidth=1.6, foreground=color)])
    for cluster_id, index in representatives.items():
        ax.scatter(embedding[index, 0], embedding[index, 1], s=44, marker="*", color=cluster_color_local(cluster_id), edgecolor="white", linewidth=0.55, zorder=5)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

def _squeezed_local(array):
    result = np.squeeze(np.asarray(array)).astype(float)
    if result.ndim != 2:
        raise ValueError(f"Expected a 2-D spectrogram, got shape {result.shape}")
    return result

def selected_spectrograms_local(data):
    representatives = data["representatives"]
    spectral = np.asarray(data["spectral"])
    reconstructed = np.asarray(data["reconstructed"])
    clusters = sorted(representatives)
    original = [_squeezed_local(spectral[representatives[cluster]]) for cluster in clusters]
    recon = [_squeezed_local(reconstructed[representatives[cluster]]) for cluster in clusters]
    residual = [np.abs(source - predicted) for source, predicted in zip(original, recon)]
    return clusters, original, recon, residual

def draw_spectrogram_local(ax, spectrogram, *, cmap, vmin, vmax, show_x, show_y):
    image = ax.imshow(
        spectrogram, origin="lower", aspect="auto", interpolation="nearest",
        cmap=cmap, vmin=vmin, vmax=vmax,
        extent=[*TIME_RANGE_S, *FREQUENCY_RANGE_HZ], rasterized=True,
    )
    ax.set_xticks([0, 30, 60] if show_x else [])
    ax.set_yticks([0, 25, 50] if show_y else [])
    if show_x:
        ax.set_xlabel("Time (s)", labelpad=1.5)
    if show_y:
        ax.set_ylabel("Frequency (Hz)", labelpad=1.5)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.45)
        spine.set_color("0.35")
    return image

def draw_main_state_plate_local(spec_axes, data):
    clusters, original, _, _ = selected_spectrograms_local(data)
    labels = np.asarray(data["labels"])
    pooled = np.concatenate([item.ravel() for item in original])
    vmin = max(0.0, float(np.nanpercentile(pooled, 1)))
    vmax = float(np.nanpercentile(pooled, 99))
    if vmax <= vmin:
        vmax = vmin + 1.0
    for column, cluster_id in enumerate(clusters):
        ax = spec_axes[column]
        count = int(np.sum(labels == cluster_id))
        draw_spectrogram_local(
            ax, original[column], cmap=MAIN_COLORS["spectrogram_cmap"],
            vmin=vmin, vmax=vmax, show_x=True, show_y=column == 0,
        )
        ax.set_title(f"C{cluster_id}  (n={count})", color=cluster_color_local(cluster_id), fontweight="bold", pad=2)
        if column == 0:
            ax.text(-0.37, 0.5, "Representative window", rotation=90, transform=ax.transAxes, ha="center", va="center", fontweight="bold")
    add_panel_label_local(spec_axes[0], "e", **MAIN_PANEL_LABEL_POS["e"])


with plt.rc_context(publication_style(**MAIN_FONT)):
    clusters = sorted(event_data["representatives"])
    fig_main = plt.figure(figsize=MAIN_FIGSIZE)
    outer = fig_main.add_gridspec(3, 2, **MAIN_GRID)

    ax_a = fig_main.add_subplot(outer[0, 0])
    grid_b = outer[0, 1].subgridspec(1, 2, **MAIN_PANEL_B)
    ax_b_curve = fig_main.add_subplot(grid_b[0, 0])
    ax_b_hist = fig_main.add_subplot(grid_b[0, 1])
    ax_c = fig_main.add_subplot(outer[1, 0])
    ax_d = fig_main.add_subplot(outer[1, 1])
    grid_e = outer[2, :].subgridspec(1, len(clusters), wspace=MAIN_PANEL_E_WSPACE)
    axes_e = [fig_main.add_subplot(grid_e[0, i]) for i in range(len(clusters))]

    draw_ae_summary_local(ax_a, train, validation)
    add_panel_label_local(ax_a, "a", **MAIN_PANEL_LABEL_POS["a"])
    draw_k_summary_local(ax_b_curve, ax_b_hist, k_curves, best_k, EVENT)
    add_panel_label_local(ax_b_curve, "b", **MAIN_PANEL_LABEL_POS["b"])
    draw_quality_summary_local(ax_c, quality, EVENT)
    add_panel_label_local(ax_c, "c", **MAIN_PANEL_LABEL_POS["c"])
    draw_tsne_local(ax_d, event_data, EVENT)
    add_panel_label_local(ax_d, "d", **MAIN_PANEL_LABEL_POS["d"])
    draw_main_state_plate_local(axes_e, event_data)

    fig_main.subplots_adjust(**MAIN_MARGINS)
    main_outputs = save_notebook_figure(
        fig_main,
        OUTPUT_DIR / "Fig2_model_result_main",
        OUTPUT_FORMATS,
        TIFF_DPI,
    )
    display(fig_main)
    plt.close(fig_main)

print("Saved main figure:")
for path in main_outputs:
    print(path)


In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from IPython.display import display

# ---------- 颜色、字体、尺寸及间距 ----------
MAIN_COLORS = {
    "ae_train": "#0F4D92", "ae_validation": "#D97706", "k_curve_highlight": "#0F4D92", "k_curve_gray": "#8F8F8F",
    "k_star": "#D97706", "k_hist_bar": "#B4C0E4", "k_hist_edge": "#0F4D92", "quality_box": "#B4C0E4",
    "quality_box_edge": "#0F4D92", "quality_star": "#D97706", "neutral_dark": "#4D4D4D", "neutral_mid": "#8F8F8F",
    "black": "#272727", "spectrogram_cmap": "magma",
    "tsne_cluster_palette": ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442", "#000000"],
}
MAIN_PANEL_LABEL_POS = {"a": dict(x=-0.14, y=1.02), "b": dict(x=-0.22, y=1.02), "c": dict(x=-0.14, y=1.02), "d": dict(x=-0.13, y=1.02), "e": dict(x=-0.27, y=1.12)}
panel_font = 14
MAIN_FIGSIZE = (7.2, 6.65)
MAIN_FONT = dict(font_family="Arial", font_size=10.0, axes_label_size=10.0, axes_title_size=10.5, tick_label_size=9.3, legend_size=9.2)
MAIN_GRID = dict(height_ratios=[0.92, 1.0, 0.76], hspace=0.35, wspace=0.28)
MAIN_PANEL_B = dict(width_ratios=[1.45, 0.75], wspace=0.42)
MAIN_PANEL_E_WSPACE = 0.08
MAIN_MARGINS = dict(left=0.08, right=0.985, top=0.97, bottom=0.08)
SAVE_DPI = 600
TIME_RANGE_S = (0.0, 60.0)
FREQUENCY_RANGE_HZ = (0.0, 50.0)
SAVE_ROOT = Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
PNG_DIR = SAVE_ROOT / "png"
TIFF_DIR = SAVE_ROOT / "tiff"

def publication_style(font_family="Arial", font_size=10.0, axes_label_size=10.0, axes_title_size=10.5, tick_label_size=9.3, legend_size=9.2):
    return {
        "font.family": font_family, "font.size": font_size, "axes.labelsize": axes_label_size, "axes.titlesize": axes_title_size,
        "xtick.labelsize": tick_label_size, "ytick.labelsize": tick_label_size, "legend.fontsize": legend_size,
        "axes.linewidth": 0.8, "axes.edgecolor": "black", "axes.labelcolor": "black", "axes.facecolor": "white", "figure.facecolor": "white",
        "axes.grid": False, "axes.spines.top": True, "axes.spines.right": True, "axes.spines.bottom": True, "axes.spines.left": True,
        "xtick.direction": "out", "ytick.direction": "out", "xtick.top": False, "ytick.right": False, "legend.frameon": False,
        "text.usetex": False, "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    }

def apply_closed_frame_local(ax, linewidth=0.8, tick_length=3.5, pad=4):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(linewidth)
    ax.tick_params(axis="both", which="both", direction="out", bottom=True, left=True, top=False, right=False, labeltop=False, labelright=False, colors="black", pad=pad)
    ax.tick_params(axis="both", which="major", length=tick_length, width=linewidth)
    ax.tick_params(axis="both", which="minor", length=2.0, width=0.6)
    ax.grid(False, which="both")

def add_panel_label_local(ax, label, x=-0.16, y=1.10):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=panel_font, fontweight="bold", ha="left", va="bottom")

def cluster_color_local(cluster_id):
    palette = MAIN_COLORS["tsne_cluster_palette"]
    return palette[int(cluster_id) % len(palette)]

def summarize_curves_local(array):
    array = np.asarray(array, dtype=float)
    return {"median": np.nanmedian(array, axis=0), "q25": np.nanpercentile(array, 25, axis=0), "q75": np.nanpercentile(array, 75, axis=0), "n": np.sum(np.isfinite(array), axis=0)}

def positive_for_log_local(values):
    values = np.asarray(values, dtype=float)
    finite_positive = values[np.isfinite(values) & (values > 0)]
    floor = np.min(finite_positive) * 0.5 if len(finite_positive) else 1e-8
    return np.where(np.isfinite(values) & (values <= 0), floor, values)

def draw_ae_summary_local(ax, train, validation):
    train_summary = summarize_curves_local(train)
    val_summary = summarize_curves_local(validation)
    for summary, color, label in ((train_summary, MAIN_COLORS["ae_train"], "Training"), (val_summary, MAIN_COLORS["ae_validation"], "Validation")):
        median = positive_for_log_local(summary["median"])
        lower = positive_for_log_local(summary["q25"])
        upper = positive_for_log_local(summary["q75"])
        epoch = np.arange(len(median))
        ax.fill_between(epoch, lower, upper, color=color, alpha=0.18, linewidth=0)
        ax.plot(epoch, median, color=color, lw=1.35, label=label)
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Reconstruction MSE")
    ax.legend(loc="upper right", frameon=False, handlelength=1.3)

def draw_k_summary_local(ax_curve, ax_hist, curves, best_k, event):
    for current_event, k_values, wcss in curves:
        k_values = np.asarray(k_values)
        wcss = np.asarray(wcss, dtype=float)
        normalized = wcss / wcss[0]
        highlighted = current_event == event
        ax_curve.plot(k_values, normalized, color=MAIN_COLORS["k_curve_highlight"] if highlighted else MAIN_COLORS["k_curve_gray"], alpha=1.0 if highlighted else 0.28, lw=1.4 if highlighted else 0.6)
        chosen = best_k[current_event]["WCSS"]
        selected = np.where(k_values == chosen)[0]
        if len(selected):
            ax_curve.scatter(chosen, normalized[selected[0]], s=58 if highlighted else 12, marker="*", color=MAIN_COLORS["k_star"], edgecolor=MAIN_COLORS["black"] if highlighted else "none", linewidth=0.35, alpha=1.0 if highlighted else 0.65, zorder=4)
    ax_curve.set_xlabel("Number of clusters, K")
    ax_curve.set_ylabel("Normalized WCSS")
    ax_curve.text(0.04, 0.08, f"{event}: K={best_k[event]['WCSS']}", transform=ax_curve.transAxes, color=MAIN_COLORS["k_curve_highlight"], fontweight="bold")
    counts = Counter(int(values["WCSS"]) for values in best_k.values())
    k_all = np.arange(min(counts), max(counts) + 1)
    heights = np.asarray([counts.get(int(k), 0) for k in k_all])
    bars = ax_hist.bar(k_all, heights, width=0.72, color=MAIN_COLORS["k_hist_bar"], edgecolor=MAIN_COLORS["k_hist_edge"], linewidth=0.6)
    for bar, value in zip(bars, heights):
        if value:
            ax_hist.text(bar.get_x() + bar.get_width() / 2, value + 0.4, str(value), ha="center", va="bottom", fontsize=8)
    ax_hist.set_xlabel("K distribution")
    ax_hist.set_ylabel("Events")
    ax_hist.set_xticks(k_all)
    ax_hist.set_ylim(0, max(heights) * 1.18)

def draw_quality_summary_local(ax, quality, event):
    values = np.asarray([row["silhouette"] for row in quality], dtype=float)
    values = values[np.isfinite(values)]
    rng = np.random.default_rng(20260812)
    x = rng.normal(1.0, 0.035, size=len(values))
    ax.boxplot(values, positions=[1], widths=0.34, patch_artist=True, showfliers=False, boxprops={"facecolor": MAIN_COLORS["quality_box"], "edgecolor": MAIN_COLORS["quality_box_edge"], "linewidth": 0.8}, medianprops={"color": MAIN_COLORS["black"], "linewidth": 1.0}, whiskerprops={"color": MAIN_COLORS["neutral_dark"], "linewidth": 0.8}, capprops={"color": MAIN_COLORS["neutral_dark"], "linewidth": 0.8})
    ax.scatter(x, values, s=10, color=MAIN_COLORS["neutral_dark"], alpha=0.55, edgecolor="none", rasterized=True)
    representative = next(row for row in quality if row["event"] == event)
    representative_value = float(representative["silhouette"])
    ax.scatter(1, representative_value, marker="*", s=72, color=MAIN_COLORS["quality_star"], edgecolor=MAIN_COLORS["black"], linewidth=0.4, zorder=5)
    ax.text(1.09, representative_value, f"{event}: {representative_value:.3f}", va="center", color=MAIN_COLORS["quality_star"], fontweight="bold")
    ax.set_xlim(0.72, 1.34)
    ax.set_xticks([1])
    ax.set_xticklabels([f"{len(values)} events"])
    ax.set_ylabel("Silhouette coefficient")

def draw_tsne_local(ax, data, event):
    embedding = np.asarray(data["tsne"])
    labels = np.asarray(data["labels"])
    representatives = data["representatives"]
    for cluster_id in sorted(int(v) for v in np.unique(labels)):
        mask = labels == cluster_id
        color = cluster_color_local(cluster_id)
        ax.scatter(embedding[mask, 0], embedding[mask, 1], s=5, alpha=0.48, color=color, edgecolor="none", rasterized=True)
        center = np.median(embedding[mask], axis=0)
        lbl = ax.text(center[0], center[1], f"C{cluster_id}", ha="center", va="center", color="white", fontsize=6.5, fontweight="bold")
        lbl.set_path_effects([path_effects.withStroke(linewidth=1.6, foreground=color)])
    for cluster_id, index in representatives.items():
        ax.scatter(embedding[index, 0], embedding[index, 1], s=44, marker="*", color=cluster_color_local(cluster_id), edgecolor="white", linewidth=0.55, zorder=5)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

def _squeezed_local(array):
    result = np.squeeze(np.asarray(array)).astype(float)
    if result.ndim != 2:
        raise ValueError(f"Expected a 2-D spectrogram, got shape {result.shape}")
    return result

def selected_spectrograms_local(data):
    representatives = data["representatives"]
    spectral = np.asarray(data["spectral"])
    reconstructed = np.asarray(data["reconstructed"])
    clusters = sorted(representatives)
    original = [_squeezed_local(spectral[representatives[cluster]]) for cluster in clusters]
    recon = [_squeezed_local(reconstructed[representatives[cluster]]) for cluster in clusters]
    residual = [np.abs(source - predicted) for source, predicted in zip(original, recon)]
    return clusters, original, recon, residual

def draw_spectrogram_local(ax, spectrogram, *, cmap, vmin, vmax, show_x, show_y):
    image = ax.imshow(spectrogram, origin="lower", aspect="auto", interpolation="nearest", cmap=cmap, vmin=vmin, vmax=vmax, extent=[*TIME_RANGE_S, *FREQUENCY_RANGE_HZ], rasterized=True)
    ax.set_xticks([0, 30, 60] if show_x else [])
    ax.set_yticks([0, 25, 50] if show_y else [])
    if show_x:
        ax.set_xlabel("Time (s)", labelpad=1.5)
    if show_y:
        ax.set_ylabel("Frequency (Hz)", labelpad=1.5)
    return image

def draw_main_state_plate_local(spec_axes, data):
    clusters, original, _, _ = selected_spectrograms_local(data)
    labels = np.asarray(data["labels"])
    pooled = np.concatenate([item.ravel() for item in original])
    vmin = max(0.0, float(np.nanpercentile(pooled, 1)))
    vmax = float(np.nanpercentile(pooled, 99))
    if vmax <= vmin:
        vmax = vmin + 1.0
    for column, cluster_id in enumerate(clusters):
        ax = spec_axes[column]
        count = int(np.sum(labels == cluster_id))
        draw_spectrogram_local(ax, original[column], cmap=MAIN_COLORS["spectrogram_cmap"], vmin=vmin, vmax=vmax, show_x=True, show_y=column == 0)
        ax.set_title(f"C{cluster_id}  (n={count})", color=cluster_color_local(cluster_id), fontweight="bold", pad=2)
        if column == 0:
            ax.text(-0.37, 0.5, "Representative window", rotation=90, transform=ax.transAxes, ha="center", va="center", fontweight="bold")
    add_panel_label_local(spec_axes[0], "e", **MAIN_PANEL_LABEL_POS["e"])

# ---------- 绘图与保存：先运行原来的数据准备代码 ----------
PNG_DIR.mkdir(parents=True, exist_ok=True)
TIFF_DIR.mkdir(parents=True, exist_ok=True)
png_path = PNG_DIR / "Fig 3.png"
tiff_path = TIFF_DIR / "Fig 3.tiff"

with plt.rc_context(publication_style(**MAIN_FONT)):
    clusters = sorted(event_data["representatives"])
    fig_main = plt.figure(figsize=MAIN_FIGSIZE)
    outer = fig_main.add_gridspec(3, 2, **MAIN_GRID)
    ax_a = fig_main.add_subplot(outer[0, 0])
    grid_b = outer[0, 1].subgridspec(1, 2, **MAIN_PANEL_B)
    ax_b_curve = fig_main.add_subplot(grid_b[0, 0])
    ax_b_hist = fig_main.add_subplot(grid_b[0, 1])
    ax_c = fig_main.add_subplot(outer[1, 0])
    ax_d = fig_main.add_subplot(outer[1, 1])
    grid_e = outer[2, :].subgridspec(1, len(clusters), wspace=MAIN_PANEL_E_WSPACE)
    axes_e = [fig_main.add_subplot(grid_e[0, i]) for i in range(len(clusters))]
    draw_ae_summary_local(ax_a, train, validation)
    add_panel_label_local(ax_a, "a", **MAIN_PANEL_LABEL_POS["a"])
    draw_k_summary_local(ax_b_curve, ax_b_hist, k_curves, best_k, EVENT)
    add_panel_label_local(ax_b_curve, "b", **MAIN_PANEL_LABEL_POS["b"])
    draw_quality_summary_local(ax_c, quality, EVENT)
    add_panel_label_local(ax_c, "c", **MAIN_PANEL_LABEL_POS["c"])
    draw_tsne_local(ax_d, event_data, EVENT)
    add_panel_label_local(ax_d, "d", **MAIN_PANEL_LABEL_POS["d"])
    draw_main_state_plate_local(axes_e, event_data)
    for ax in (ax_a, ax_b_curve, ax_b_hist, ax_c, ax_d):
        apply_closed_frame_local(ax, linewidth=0.8, tick_length=3.5, pad=4)
    for ax in axes_e:
        apply_closed_frame_local(ax, linewidth=0.6, tick_length=2.5, pad=2)
    fig_main.subplots_adjust(**MAIN_MARGINS)
    fig_main.savefig(png_path, format="png", dpi=SAVE_DPI, facecolor="white", transparent=False, bbox_inches="tight", pad_inches=0)
    fig_main.savefig(tiff_path, format="tiff", dpi=SAVE_DPI, facecolor="white", transparent=False, bbox_inches="tight", pad_inches=0, pil_kwargs={"compression": "tiff_lzw"})
    display(fig_main)
    plt.close(fig_main)

print("Saved Fig 3:")
print(png_path)
print(tiff_path)

# figS2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

# ---------- 可编辑：颜色、字体、尺寸及间距 ----------
SI_D_COLORS = {
    "background_trajectories": "#8F8F8F",
    "total": "#0F4D92",
    "cluster_weighted": "tab:red",
    "reconstruction": "#9A7CC1",
    "neutral_dark": "#4D4D4D",
}

SI_D_PANEL_LABEL_POS = dict(x=-0.19, y=1.10)
SI_D_PANEL_LABEL_ENABLED = False
SI_D_LABEL = "d"

SI_D_FIGSIZE = (3.6, 3.0)

SI_D_FONT = dict(
    font_family="Arial",
    font_size=10.0,
    axes_label_size=10.0,
    axes_title_size=10.5,
    tick_label_size=9.3,
    legend_size=9.2,
)

SI_D_MARGINS = dict(
    left=0.18,
    right=0.96,
    top=0.94,
    bottom=0.20,
)

SI_D_OUTPUT_FORMATS = ("svg", "pdf", "tiff", "png")
SI_D_TIFF_DPI = 600
GAMMA_LOCAL = 0.1


def summarize_curves_local(array):
    return {
        "median": np.nanmedian(array, axis=0),
        "q25": np.nanpercentile(array, 25, axis=0),
        "q75": np.nanpercentile(array, 75, axis=0),
    }


def draw_dec_loss_local(ax, loss_summary):
    progress = np.asarray(loss_summary["progress"], dtype=float)
    aligned = loss_summary["aligned"]
    summaries = loss_summary["summaries"]

    # 各事件的总损失轨迹
    for values in aligned["total"]:
        ax.plot(
            progress,
            values,
            color=SI_D_COLORS["background_trajectories"],
            alpha=0.10,
            lw=0.38,
            zorder=1,
        )

    # 中位数曲线及四分位区间
    series = [
        ("total", SI_D_COLORS["total"], "Total"),
        (
            "cluster_weighted",
            SI_D_COLORS["cluster_weighted"],
            rf"{GAMMA_LOCAL:g} $\times$ KL",
        ),
        (
            "reconstruction",
            SI_D_COLORS["reconstruction"],
            "Reconstruction",
        ),
    ]

    for component, color, label in series:
        summary = summaries[component]

        ax.fill_between(
            progress,
            summary["q25"],
            summary["q75"],
            color=color,
            alpha=0.14,
            linewidth=0,
            zorder=2,
        )

        ax.plot(
            progress,
            summary["median"],
            color=color,
            lw=1.35,
            label=label,
            zorder=3,
        )

    # 坐标轴
    ax.set_xlabel("Normalized training progress (%)")
    ax.set_ylabel("Weighted loss")
    ax.set_xlim(0, 100)
    ax.set_xticks(np.arange(0, 101, 20))
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
    # 四边封闭边框
    ax.set_facecolor("white")
    for side in ("left", "right", "bottom", "top"):
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("black")
        ax.spines[side].set_linewidth(0.8)

    # 四边刻度向内，上侧和右侧不显示刻度标签
    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=3.5,
        width=0.8,
        colors="black",
        bottom=True,
        top=False,
        left=True,
        right=False,
        labelbottom=True,
        labeltop=False,
        labelleft=True,
        labelright=False,
        pad=4,
    )

    ax.minorticks_off()
    ax.grid(False, which="both")

    # 无边框图例
    ax.legend(
        loc="upper right",
        frameon=False,
        handlelength=1.5,
        handletextpad=0.5,
        labelspacing=0.35,
        borderaxespad=0.6,
    )

    # 损失函数
    ax.text(
        0.03,
        0.94,
        rf"$L_{{\mathrm{{total}}}}"
        rf"={GAMMA_LOCAL:g}L_{{\mathrm{{KL}}}}"
        rf"+L_{{\mathrm{{recon}}}}$",
        transform=ax.transAxes,
        fontsize=8.5,
        color=SI_D_COLORS["neutral_dark"],
        ha="left",
        va="top",
    )


# ---------- 绘图、保存及显示 ----------
with plt.rc_context(publication_style(**SI_D_FONT)):
    fig_si_d, ax_si_d = plt.subplots(figsize=SI_D_FIGSIZE)
    fig_si_d.patch.set_facecolor("white")

    draw_dec_loss_local(ax_si_d, dec_loss_summary)

    if SI_D_PANEL_LABEL_ENABLED:
        ax_si_d.text(
            SI_D_PANEL_LABEL_POS["x"],
            SI_D_PANEL_LABEL_POS["y"],
            SI_D_LABEL,
            transform=ax_si_d.transAxes,
            fontsize=11,
            fontweight="bold",
            ha="left",
            va="bottom",
        )

    fig_si_d.subplots_adjust(**SI_D_MARGINS)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    si_d_outputs = save_notebook_figure(
        fig_si_d,
        OUTPUT_DIR / "FigS2d_DEC_loss",
        SI_D_OUTPUT_FORMATS,
        SI_D_TIFF_DPI,
    )

    display(fig_si_d)
    plt.close(fig_si_d)

print("Saved supplementary panel d:")
for path in si_d_outputs:
    print(path)

In [ ]:
from pathlib import Path

# ---------- 保存路径 ----------
SAVE_ROOT = Path(
    r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben"
    r"\uilt_oneday_all\uilt_paper_figure\Fig_all"
)

TIFF_DIR = SAVE_ROOT / "tiff"
PNG_DIR = SAVE_ROOT / "png"

TIFF_DIR.mkdir(parents=True, exist_ok=True)
PNG_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 绘图、保存及显示 ----------
with plt.rc_context(publication_style(**SI_D_FONT)):
    fig_si_d, ax_si_d = plt.subplots(figsize=SI_D_FIGSIZE)
    fig_si_d.patch.set_facecolor("white")

    draw_dec_loss_local(ax_si_d, dec_loss_summary)

    if SI_D_PANEL_LABEL_ENABLED:
        ax_si_d.text(
            SI_D_PANEL_LABEL_POS["x"],
            SI_D_PANEL_LABEL_POS["y"],
            SI_D_LABEL,
            transform=ax_si_d.transAxes,
            fontsize=11,
            fontweight="bold",
            ha="left",
            va="bottom",
        )

    fig_si_d.subplots_adjust(**SI_D_MARGINS)

    tiff_path = TIFF_DIR / "Fig S2.tiff"
    png_path = PNG_DIR / "Fig S2.png"

    fig_si_d.savefig(
        tiff_path,
        format="tiff",
        dpi=600,
        facecolor="white",
        transparent=False,
        pil_kwargs={"compression": "tiff_lzw"},    bbox_inches="tight",

    )

    fig_si_d.savefig(
        png_path,
        format="png",
        dpi=600,
        facecolor="white",
        transparent=False,
            bbox_inches="tight"
    )

    display(fig_si_d)
    plt.close(fig_si_d)

print("Saved:")
print(tiff_path)
print(png_path)

## Supplementary Figure panel e — reconstruction audit (separate figure)


In [ ]:
# ---------- Editable: colours, panel-label positions, typography, dimensions and spacing ----------
SI_E_COLORS = {
    "cluster_palette": [
        "#0072B2", "#E69F00", "#009E73", "#CC79A7",
        "#56B4E9", "#D55E00", "#F0E442", "#000000",
    ],
    "amplitude_cmap":   "magma",
    "residual_cmap":    "viridis",
    "text_on_spectrogram": "white",
    "text_bbox_face":   "black",
}
SI_E_PANEL_LABEL_POS = dict(x=-0.34, y=1.25)
SI_E_PANEL_LABEL_ENABLED = False
SI_E_PANEL_LABEL = "e"
SI_E_TITLE = "Spectrogram reconstruction audit"
SI_E_FIGSIZE = (7.2, 5.25)
SI_E_FONT = dict(
    font_family="Arial",
    font_size=10.0,
    axes_label_size=10.0,
    axes_title_size=10.5,
    tick_label_size=9.3,
    legend_size=9.2,
)
fontrms = 9
SI_E_HEIGHT_RATIOS = [1.0, 1.0, 1.0, 0.09]
SI_E_HSPACE = 0.25
SI_E_WSPACE = 0.08
SI_E_MARGINS = dict(left=0.08, right=0.985, top=0.93, bottom=0.08)
SI_E_OUTPUT_FORMATS = ("svg", "pdf", "tiff", "png")
SI_E_TIFF_DPI = 600
TIME_RANGE_S = (0.0, 60.0)
FREQUENCY_RANGE_HZ = (0.0, 50.0)
# --------------------------------------------------------------------------------------------------------

def cluster_color_local(cluster_id):
    return SI_E_COLORS["cluster_palette"][cluster_id % len(SI_E_COLORS["cluster_palette"])]

def _squeezed_local(array):
    result = np.squeeze(np.asarray(array)).astype(float)
    if result.ndim != 2:
        raise ValueError(f"Expected a 2-D spectrogram, got shape {result.shape}")
    return result

def selected_spectrograms_local(data):
    representatives = data["representatives"]
    spectral = np.asarray(data["spectral"])
    reconstructed = np.asarray(data["reconstructed"])
    clusters = sorted(representatives)
    original = [_squeezed_local(spectral[representatives[c]]) for c in clusters]
    recon = [_squeezed_local(reconstructed[representatives[c]]) for c in clusters]
    residual = [np.abs(s - p) for s, p in zip(original, recon)]
    return clusters, original, recon, residual

def draw_spectrogram_local(ax, spectrogram, *, cmap, vmin, vmax, show_x, show_y):
    image = ax.imshow(
        spectrogram, origin="lower", aspect="auto", interpolation="nearest",
        cmap=cmap, vmin=vmin, vmax=vmax,
        extent=[*TIME_RANGE_S, *FREQUENCY_RANGE_HZ], rasterized=True,
    )
    ax.set_xticks([0, 30, 60] if show_x else [])
    ax.set_yticks([0, 25, 50] if show_y else [])
    if show_x:
        ax.set_xlabel("Time (s)", labelpad=1.5)
    if show_y:
        ax.set_ylabel("Frequency (Hz)", labelpad=1.5)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.80)
        spine.set_color("black")
    ax.tick_params(axis="both",which="both",direction="out",top=False,right=False,bottom=True,left=True,length=3,width=0.8)    
    return image

def pearson_correlation_local(first, second):
    a = np.asarray(first, dtype=float).ravel()
    b = np.asarray(second, dtype=float).ravel()
    a = a - np.mean(a)
    b = b - np.mean(b)
    denom = np.sqrt(np.sum(a * a) * np.sum(b * b))
    if denom <= 0:
        return np.nan
    return float(np.sum(a * b) / denom)

def reconstruction_metrics_local(original, reconstructed):
    residual = original - reconstructed
    ov = float(np.var(original))
    return {
        "rmse": float(np.sqrt(np.mean(residual * residual))),
        "pixel_correlation": pearson_correlation_local(original, reconstructed),
        "frequency_profile_correlation": pearson_correlation_local(np.mean(original, axis=1), np.mean(reconstructed, axis=1)),
        "time_profile_correlation":      pearson_correlation_local(np.mean(original, axis=0), np.mean(reconstructed, axis=0)),
        "variance_retention": (float(np.var(reconstructed)) / ov) if ov > 0 else np.nan,
    }

def draw_reconstruction_audit_local(
    fig, original_axes, reconstruction_axes, residual_axes,
    amplitude_colorbar_ax, residual_colorbar_ax, data,
):
    clusters, original, reconstructed, residual = selected_spectrograms_local(data)
    labels = np.asarray(data["labels"])
    amplitude_pool = np.concatenate([v.ravel() for v in original + reconstructed])
    amplitude_vmin = max(0.0, float(np.nanpercentile(amplitude_pool, 1)))
    amplitude_vmax = float(np.nanpercentile(amplitude_pool, 99))
    residual_vmax = float(np.nanpercentile(np.concatenate([v.ravel() for v in residual]), 99))

    amplitude_image = None
    residual_image = None
    for column, cluster_id in enumerate(clusters):
        metrics = reconstruction_metrics_local(original[column], reconstructed[column])
        count = int(np.sum(labels == cluster_id))
        amplitude_image = draw_spectrogram_local(
            original_axes[column], original[column],
            cmap=SI_E_COLORS["amplitude_cmap"], vmin=amplitude_vmin, vmax=amplitude_vmax,
            show_x=False, show_y=column == 0,
        )
        draw_spectrogram_local(
            reconstruction_axes[column], reconstructed[column],
            cmap=SI_E_COLORS["amplitude_cmap"], vmin=amplitude_vmin, vmax=amplitude_vmax,
            show_x=False, show_y=column == 0,
        )
        residual_image = draw_spectrogram_local(
            residual_axes[column], residual[column],
            cmap=SI_E_COLORS["residual_cmap"], vmin=0.0, vmax=residual_vmax,
            show_x=True, show_y=column == 0,
        )
        residual_axes[column].set_xlabel("")
        original_axes[column].set_title(
            f"C{cluster_id}  (n={count})",
            color=cluster_color_local(cluster_id), fontweight="bold", pad=2,
        )
        reconstruction_axes[column].text(
            0.03, 0.95, rf"$r_{{freq}}$={metrics['frequency_profile_correlation']:.2f}",
            transform=reconstruction_axes[column].transAxes, ha="left", va="top",
            fontsize=fontrms, color=SI_E_COLORS["text_on_spectrogram"],
            bbox={"facecolor": SI_E_COLORS["text_bbox_face"], "edgecolor": "none", "alpha": 0.45, "pad": 1.0},
        )
        residual_axes[column].set_title(f"RMSE={metrics['rmse']:.3f}", fontsize=fontrms, pad=2)

    residual_axes[-1].text(
        0.97, 0.04, "Time (s)",
        transform=residual_axes[-1].transAxes, ha="right", va="bottom", fontsize=7,
        color=SI_E_COLORS["text_on_spectrogram"],
        bbox={"facecolor": SI_E_COLORS["text_bbox_face"], "edgecolor": "none", "alpha": 0.35, "pad": 0.8},
    )

    row_specs = (
        (original_axes[0],       "Original"),
        (reconstruction_axes[0], "Reconstruction"),
        (residual_axes[0],       "|Residual|"),
    )
    for ax, label_text in row_specs:
        ax.text(-0.34, 0.5, label_text, rotation=90, transform=ax.transAxes, ha="center", va="center", fontweight="bold")

    if SI_E_PANEL_LABEL_ENABLED:
        original_axes[0].text(
            SI_E_PANEL_LABEL_POS["x"], SI_E_PANEL_LABEL_POS["y"], SI_E_PANEL_LABEL,
            transform=original_axes[0].transAxes, fontsize=9, fontweight="bold", ha="left", va="bottom",
        )



    amp_cb = fig.colorbar(amplitude_image, cax=amplitude_colorbar_ax, orientation="horizontal")
    amp_cb.set_label("Normalized amplitude", labelpad=2)
    res_cb = fig.colorbar(residual_image, cax=residual_colorbar_ax, orientation="horizontal")
    res_cb.set_label("|Residual|", labelpad=2)


with plt.rc_context(publication_style(**SI_E_FONT)):
    clusters = sorted(event_data["representatives"])
    fig_si_e = plt.figure(figsize=SI_E_FIGSIZE)
    audit_grid = fig_si_e.add_gridspec(
        4, len(clusters), height_ratios=SI_E_HEIGHT_RATIOS,
        hspace=SI_E_HSPACE, wspace=SI_E_WSPACE,
    )
    original_axes = [fig_si_e.add_subplot(audit_grid[0, i]) for i in range(len(clusters))]
    reconstruction_axes = [fig_si_e.add_subplot(audit_grid[1, i]) for i in range(len(clusters))]
    residual_axes = [fig_si_e.add_subplot(audit_grid[2, i]) for i in range(len(clusters))]
    split = max(1, len(clusters) // 2)
    amplitude_colorbar_ax = fig_si_e.add_subplot(audit_grid[3, :split])
    residual_colorbar_ax = fig_si_e.add_subplot(audit_grid[3, split:])

    draw_reconstruction_audit_local(
        fig_si_e,
        original_axes, reconstruction_axes, residual_axes,
        amplitude_colorbar_ax, residual_colorbar_ax,
        event_data,
    )
    fig_si_e.subplots_adjust(**SI_E_MARGINS)

    si_e_outputs = save_notebook_figure(
        fig_si_e,
        OUTPUT_DIR / "FigS2e_reconstruction_audit",
        SI_E_OUTPUT_FORMATS,
        SI_E_TIFF_DPI,
    )
    display(fig_si_e)
    
    fig_si_e.savefig(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all\tiff\Fig S3.tiff", format="tiff", dpi=600, facecolor="white", transparent=False, pil_kwargs={"compression": "tiff_lzw"}, bbox_inches="tight", pad_inches=0)
    fig_si_e.savefig(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all\png\Fig S3.png", format="png", dpi=600, facecolor="white", transparent=False, bbox_inches="tight", pad_inches=0)
    
    plt.close(fig_si_e)

print("Saved supplementary panel e:")
for path in si_e_outputs:
    print(path)
